# 02 — Diagnostyka ladunkow `W` (bez ground-truth)

Odpowiednik `w_loading_tests`, ale **bez prawdziwego `W`**: zamiast porownywac do
ground-truth, charakteryzujemy **to, czego model sie nauczyl** i jak bardzo jest to
**powtarzalne**. Korzystamy tylko z wyjsc modelu (`E[W]`, ARD `E[alpha]`, rekonstrukcja)
i etykiet kohort.

| Test | Co liczymy | Po co |
|------|------------|-------|
| 1 | Heatmapy `E[W]` per widok | jaka strukture ladunkow znalazl model |
| 2 | Istotnosc czynnika: energia `||W[:,k]||`, ARD `1/E[alpha]`, `var(Z)` | ktore czynniki sa "zywe" |
| 3 | Jakosc rekonstrukcji `R^2` i korelacje per cecha | czy `Z*W^T` wyjasnia dane |
| 4 | Powtarzalnosc `E[W]` miedzy ziarnami (`|corr|` po Hungarianie) | czy rozwiazanie jest stabilne |
| 5 | `R^2` i liczba istotnych czynnikow vs `K` | ile czynnikow potrzeba |

**Uwaga o rzadkosci.** W tym modelu wskaznik spike-and-slab `E[S]` (`s_m_node.vi_gamma`)
pozostaje praktycznie rowny 1, wiec nie niesie informacji o rzadkosci — istotnosc
czynnikow oceniamy wiec energia ladunkow `||W[:,k]||` oraz precyzja ARD `E[alpha]`
(male `alpha` = czynnik istotny, duze = wygaszony przez ARD).

## Kontrakt danych
- `views` (Views z cohorts), `K`. Reszta wyprowadzana automatycznie.

## Komorka 1 — wczytanie danych

In [ ]:
# =====================================================================
#  KOMORKA 1 — WCZYTANIE DANYCH
# =====================================================================
import os, sys, warnings
# Znajdz katalog zawierajacy pakiet `src` (dziala z final_notebooks/ i z roota repo).
_p = os.path.abspath('')
for _ in range(4):
    if os.path.isdir(os.path.join(_p, 'src')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
    _p = os.path.dirname(_p)
warnings.filterwarnings('ignore')

import numpy as np
from src.views import Views


def make_synthetic_cohort_data(C=4, K=4, n_active=2, n_per_cohort=120,
                               dims=(20, 15), sigma=0.6, noise=1.0,
                               mu_scale=1.9, block_sparse_W=False, seed=0):
    """Domyslne dane DEMO (do podmiany na wlasne).

    C kohort o roznych profilach mu_c w przestrzeni K czynnikow; tylko pierwsze
    `n_active` czynnikow rozni kohorty (reszta wspolna -- "cicha"). Widoki powstaja
    jako Y_m = Z W_m^T + szum. Zwraca (views, codes); NIE zwraca ground-truth --
    cala analiza korzysta tylko z `views` i etykiet kohort.
    """
    rng = np.random.default_rng(seed)
    mu = np.zeros((C, K))
    mu[:, :n_active] = rng.normal(scale=mu_scale, size=(C, n_active))
    codes = np.repeat(np.arange(C), n_per_cohort)
    Z = np.array([rng.normal(mu[c], sigma) for c in codes])
    Ys = []
    for d in dims:
        if block_sparse_W:
            W = np.zeros((d, K))                              # prawdziwe zera poza blokiem
            for k, idx in enumerate(np.array_split(np.arange(d), K)):
                W[idx, k] = rng.normal(loc=1.5, scale=0.25, size=len(idx))  # blok cech -> czynnik k
        else:
            W = rng.normal(size=(d, K))
        Ys.append(Z @ W.T + rng.normal(scale=noise, size=(Z.shape[0], d)))
    cohorts = np.array([f'c{c}' for c in codes])
    return Views.from_list(Ys, cohorts=cohorts), codes


# >>>>>>>>>>>>>>>>>>>>>>>  PODMIEN TE SEKCJE NA SWOJE DANE  >>>>>>>>>>>>>>>>>>>>>>>
# Wymagane: `views` (Views z cohorts) oraz `K`.
views, _ = make_synthetic_cohort_data(C=4, K=4, n_active=2, seed=0)
K = 4
# <<<<<<<<<<<<<<<<<<<<<<<  KONIEC SEKCJI DO PODMIANY  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<

# Wyprowadzenie etykiet kohort z danych (dziala tez dla Twoich danych):
assert views.cohorts is not None, "views musi miec ustawione cohorts (etykieta kohorty per probka)."
cohort_names, codes = np.unique(views.cohorts, return_inverse=True)
C = len(cohort_names)
factors = [f'Z{k}' for k in range(K)]
print(f"N = {views.N} probek | widoki (liczba cech): {[v.D for v in views.simple]}")
print(f"C = {C} kohort: {list(cohort_names)} | K = {K} czynnikow do dopasowania")

## Setup — model i helpery

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import FACTModel
from src.enums import Likelihood, WPrior, ZPrior
from src.model_config import CohortPriorConfig, ModelConfig, SimpleViewConfig
from scipy.optimize import linear_sum_assignment

def fit_cohort(views, K, pi=0.5, max_iter=100, seed=0):
    """Dopasuj CohortFACTM: WPrior.ARD_SS + ZPrior.COHORT, prior spike-and-slab `pi`."""
    cfg = ModelConfig(
        simple_view_configs=[SimpleViewConfig(likelihood=Likelihood.NORMAL, w_prior=WPrior.ARD_SS)
                             for _ in range(views.num_simple)],
        structured_view_configs=[],
        z_priors=[ZPrior.COHORT] * K,
        cohort_prior_config=CohortPriorConfig(pi=pi),
    )
    m = FACTModel(views=views, K=K, model_config=cfg, seed=seed)
    m.fit(max_iter=max_iter, pretrain=True, elbo_tres=0.0)
    return m

def get_W(model, view_idx=0):
    """Macierz ladunkow E[W] (D x K) dla widoku prostego."""
    return np.asarray(model.fa.nodelist_w[view_idx].E_w)


def factor_alpha(model, view_idx=0):
    """ARD precyzja E[alpha] per czynnik (male = istotny, duze = wygaszony).
    Fallback do 1/energia, gdy wezel ARD nie wystawia E_alpha."""
    wn = model.fa.nodelist_w[view_idx]
    if hasattr(wn, 'alpha_m_node') and hasattr(wn.alpha_m_node, 'E_alpha'):
        return np.asarray(wn.alpha_m_node.E_alpha)
    e = np.linalg.norm(np.asarray(wn.E_w), axis=0)
    return 1.0 / np.maximum(e, 1e-9)


def reconstruct(model, views, view_idx=0):
    """Zwraca (Y, Y_hat) gdzie Y_hat = Z * W^T (rekonstrukcja widoku z modelu)."""
    Z = np.asarray(model.get_latent_factors())
    W = get_W(model, view_idx)
    Y = np.asarray(views.simple[view_idx].data)
    return Y, Z @ W.T


def r2_view(Y, Yh):
    """Wspolczynnik determinacji R^2 calego widoku (1 = idealna rekonstrukcja)."""
    return 1.0 - ((Y - Yh) ** 2).sum() / ((Y - Y.mean(0)) ** 2).sum()


def n_relevant_factors(model, view_idx=0, rel_thr=0.05):
    """Liczba czynnikow o wzglednej energii ladunkow > rel_thr * max."""
    e = np.linalg.norm(get_W(model, view_idx), axis=0)
    return int((e / e.max() > rel_thr).sum()) if e.max() > 0 else 0


def abscorr(a, b):
    """|corr| odporne na stale kolumny (zwraca nan zamiast bledu)."""
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return abs(np.corrcoef(a, b)[0, 1])


def align_factors(Z_ref, Z_other):
    """Hungarian na Z: perm, signs t. ze Z_other[:, perm] * signs ~ Z_ref. Bez ground-truth."""
    K = Z_ref.shape[1]
    Z_ref, Z_other = np.asarray(Z_ref), np.asarray(Z_other)
    corr = np.nan_to_num(np.array([[np.corrcoef(Z_ref[:, i], Z_other[:, j])[0, 1]
                                    for j in range(K)] for i in range(K)]))
    row, col = linear_sum_assignment(-np.abs(corr))
    signs = np.sign(corr[row, col]); signs[signs == 0] = 1
    return col, signs.astype(float)

## Test 1 — nauczone ladunki `E[W]`

Heatmapa macierzy ladunkow (cechy x czynniki) dla kazdego widoku prostego — pokazuje,
jaka strukture zaleznosci cech od czynnikow utajonych znalazl model.

In [ ]:
m = fit_cohort(views, K=K, pi=0.5, seed=0)
n_views = views.num_simple
fig, axes = plt.subplots(1, n_views, figsize=(5.2 * n_views, 0.5 + 0.18 * max(v.D for v in views.simple)), squeeze=False)
for v in range(n_views):
    E_w = get_W(m, v)
    vmax = np.abs(E_w).max()
    sns.heatmap(E_w, cmap='coolwarm', center=0, vmin=-vmax, vmax=vmax,
                xticklabels=factors, yticklabels=False, ax=axes[0, v])
    axes[0, v].set_title(f'widok {v}: E[W] (cechy x czynniki)'); axes[0, v].set_ylabel('cechy')
plt.tight_layout(); plt.show()

## Test 2 — istotnosc czynnikow

Trzy spojne miary, ktore czynniki sa "zywe": energia ladunkow `||W[:,k]||`, istotnosc
ARD `1/E[alpha]` (wieksza = istotniejszy) oraz wariancja czynnika `var(Z[:,k])`.
Czynniki o niskiej energii i wysokim `alpha` zostaly wygaszone przez ARD.

In [ ]:
W0 = get_W(m, 0)
energy = np.linalg.norm(W0, axis=0)
relevance = 1.0 / factor_alpha(m, 0)
zvar = np.asarray(m.get_latent_factors()).var(0)
tab = pd.DataFrame({'||W[:,k]|| (energia)': energy.round(3),
                    '1/E[alpha] (istotnosc ARD)': relevance.round(3),
                    'var(Z[:,k])': zvar.round(3)}, index=factors)
print(tab.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].bar(factors, energy, color='steelblue'); axes[0].set_title('Energia ladunkow ||W[:,k]||')
axes[1].bar(factors, relevance, color='seagreen'); axes[1].set_title('Istotnosc ARD: 1/E[alpha]')
axes[2].bar(factors, zvar, color='slateblue'); axes[2].set_title('Wariancja czynnika var(Z[:,k])')
for ax in axes:
    ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## Test 3 — jakosc rekonstrukcji `Y ~ Z * W^T`

Najbardziej bezposredni, beznadzorowany dowod jakosci ladunkow: jak dobrze `Z*W^T`
odtwarza obserwowane dane. `R^2` blisko 1 = ladunki dobrze tlumacza widok (reszta to szum).
Histogram pokazuje korelacje rekonstrukcji per cecha; scatter — najlepiej odtworzona cecha.

In [ ]:
n_views = views.num_simple
fig, axes = plt.subplots(1, n_views + 1, figsize=(5 * n_views + 4, 3.8), squeeze=False)
axes = axes[0]
r2_all = []
for vi in range(n_views):
    Y, Yh = reconstruct(m, views, vi)
    r2 = r2_view(Y, Yh); r2_all.append(r2)
    per_feat = np.array([abscorr(Y[:, j], Yh[:, j]) for j in range(Y.shape[1])])
    axes[vi].hist(per_feat[~np.isnan(per_feat)], bins=15, color='steelblue', alpha=0.85)
    axes[vi].set_title(f'widok {vi}: R2 = {r2:.3f}')
    axes[vi].set_xlabel('corr(Y, Y_hat) per cecha'); axes[vi].grid(alpha=0.3, axis='y')

Y, Yh = reconstruct(m, views, 0)
j = int(np.nanargmax([abscorr(Y[:, jj], Yh[:, jj]) for jj in range(Y.shape[1])]))
axes[-1].scatter(Y[:, j], Yh[:, j], s=8, alpha=0.5)
lim = [min(Y[:, j].min(), Yh[:, j].min()), max(Y[:, j].max(), Yh[:, j].max())]
axes[-1].plot(lim, lim, 'r--', lw=1)
axes[-1].set_title(f'widok 0, cecha {j}: Y vs Y_hat'); axes[-1].set_xlabel('Y'); axes[-1].set_ylabel('Y_hat')
axes[-1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Mean R2 (widoki):', round(float(np.mean(r2_all)), 3))

## Test 4 — powtarzalnosc ladunkow miedzy ziarnami

Bez ground-truth najlepszym dowodem jakosci jest **stabilnosc**: dopasowujemy kilka
modeli z roznymi ziarnami, wyrownujemy czynniki Hungarianem (na `Z`) i liczymy `|corr|`
kolumn `E[W]` wzgledem modelu referencyjnego. Wysokie `|corr|` = rozwiazanie powtarzalne
(nie artefakt jednego startu).

In [ ]:
seeds = [0, 1, 2, 3, 4]
m_ref = fit_cohort(views, K=K, pi=0.5, seed=seeds[0])
Z_ref = m_ref.get_latent_factors()
W_ref = get_W(m_ref, 0)

stab_W = []
for s in seeds[1:]:
    ms = fit_cohort(views, K=K, pi=0.5, seed=s)
    perm, signs = align_factors(Z_ref, ms.get_latent_factors())
    Ws = get_W(ms, 0)[:, perm] * signs[None, :]
    stab_W.append([abscorr(W_ref[:, k], Ws[:, k]) for k in range(K)])
stab_W = np.array(stab_W)

dfst = pd.DataFrame({'mean |corr| E[W] vs ref': np.nanmean(stab_W, 0).round(3),
                     'std |corr|': np.nanstd(stab_W, 0).round(3)}, index=factors)
print(f"Powtarzalnosc wzgledem modelu referencyjnego (seed={seeds[0]}), {len(seeds) - 1} powtorzen:")
print(dfst.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(factors, np.nanmean(stab_W, 0), yerr=np.nanstd(stab_W, 0), capsize=4, color='steelblue')
ax.set_ylim(0, 1.05); ax.set_ylabel('|corr| E[W] wzgledem referencji')
ax.set_title('Powtarzalnosc ladunkow miedzy ziarnami'); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## Test 5 — ile czynnikow potrzeba? (`R^2` i istotne czynniki vs `K`)

Dopasowujemy model dla roznych `K` i patrzymy, jak rosnie jakosc rekonstrukcji oraz ile
czynnikow pozostaje istotnych (energia > 5% maksimum). "Lokiec" na krzywej `R^2` sugeruje
sensowna liczbe czynnikow.

In [ ]:
K_grid = [2, 3, 4, 5, 6]
rows = []
for Kk in K_grid:
    mk = fit_cohort(views, K=Kk, pi=0.5, seed=0)
    r2s = [r2_view(*reconstruct(mk, views, vi)) for vi in range(views.num_simple)]
    rows.append({'K': Kk, 'mean R2': float(np.mean(r2s)),
                 'n_relevant (v0)': n_relevant_factors(mk, 0),
                 'elbo': float(mk.elbo_sequence[-1])})
dfk = pd.DataFrame(rows)
print(dfk.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(dfk['K'], dfk['mean R2'], 'o-')
axes[0].set_xlabel('K'); axes[0].set_ylabel('mean R2 (widoki)')
axes[0].set_title('Jakosc rekonstrukcji vs liczba czynnikow K'); axes[0].grid(alpha=0.3)
axes[1].plot(dfk['K'], dfk['n_relevant (v0)'], 'o-', color='C3')
axes[1].plot(dfk['K'], dfk['K'], ':', color='gray', alpha=0.6, label='y = K')
axes[1].set_xlabel('K'); axes[1].set_ylabel('liczba istotnych czynnikow (widok 0)')
axes[1].set_title('Istotne czynniki (energia > 5% max) vs K'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()